# DINOv3 + EoMT for TWSI Segmentation
---

This notebook fine-tunes an Encoder-only Mask Transformer (EoMT) semantic segmentation
model using DINOv3 as the backbone feature extractor, via the `lightly-train` library.

The EoMT model uses 100 learnable queries and an attention-mask annealing strategy
for state-of-the-art segmentation performance.

**Paper Reference:** GuideTWSI (ICRA 2026), Section VI-A

## 1. Setup

Install lightly-train with DINOv3 + EoMT support.

In [ ]:
!pip install lightly-train>=0.11.2 -q

In [ ]:
import lightly_train

## 2. Configuration

Load training hyperparameters from the config file.

**Important:** To use DINOv3, sign up and accept the terms of use from Meta to obtain
checkpoint download links.

In [ ]:
import yaml

with open("../configs/dinov3_eomt.yaml", "r") as f:
    config = yaml.safe_load(f)

print("Config loaded:")
for section, values in config.items():
    print(f"  {section}: {values}")

In [ ]:
# DINOv3 checkpoint URL from Meta (requires sign-up)
dinov3_model_url = ""  # Add your download link here

# Extract config values
IMAGE_SIZE = tuple(config["training"]["image_size"])  # (518, 518)
NORMALIZE_MEAN = config["training"]["normalize_mean"]
NORMALIZE_STD = config["training"]["normalize_std"]
LIGHTLY_MODEL = config["model"]["lightly_model"]  # dinov3/vits16-eomt

## 3. Download Dataset

Download the GuideTWSI dataset from HuggingFace Hub.

In [ ]:
# Download dataset from HuggingFace
# !huggingface-cli download guidedogrobot-tactile/GuideTWSI --local-dir ./data

# Set data paths
# Expected structure:
# data/train/images/*.jpg + data/train/masks/*.png
# data/val/images/*.jpg + data/val/masks/*.png
TRAIN_IMAGES = "data/train/images"
TRAIN_MASKS = "data/train/masks"
VAL_IMAGES = "data/val/images"
VAL_MASKS = "data/val/masks"
TEST_IMAGES = "data/test/images"
TEST_MASKS = "data/test/masks"
OUTPUT_DIR = "out/dinov3_eomt"

## 4. Training

Fine-tune the DINOv3+EoMT model using lightly-train's semantic segmentation API.

**Notes:**
- Training the background class as well is recommended for best results.
- Multi-GPU training is supported and recommended for faster convergence.
- TensorBoard logs are saved in the output directory.
- Documentation: [lightly-train API](https://docs.lightly.ai/train/stable/python_api/lightly_train.html)

In [ ]:
if __name__ == "__main__":
    lightly_train.train_semantic_segmentation(
        out=OUTPUT_DIR,
        model=LIGHTLY_MODEL,
        model_args={
            "backbone_url": dinov3_model_url,
        },
        data={
            "train": {
                "images": TRAIN_IMAGES,
                "masks": TRAIN_MASKS,
            },
            "val": {
                "images": VAL_IMAGES,
                "masks": VAL_MASKS,
            },
            "classes": {
                0: "background",
                255: "tactile-paving",
            },
        },
        resume_interrupted=True,
        transform_args={
            "image_size": IMAGE_SIZE,
            "normalize": {
                "mean": NORMALIZE_MEAN,
                "std": NORMALIZE_STD,
            },
        },
    )

## 5. Inference

Load the trained model and run inference on a test image.

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Load trained checkpoint
checkpoint_path = os.path.join(OUTPUT_DIR, "checkpoints", "last.ckpt")  # Update as needed
model = lightly_train.load_model_from_checkpoint(checkpoint_path)
print("Model loaded successfully.")

In [ ]:
# Select a test image
test_images_list = sorted(os.listdir(TEST_IMAGES))
test_image_fpath = os.path.join(TEST_IMAGES, test_images_list[0])
test_mask_fpath = os.path.join(TEST_MASKS, test_images_list[0].replace(".jpg", ".png"))

test_image = Image.open(test_image_fpath)
gt_mask = Image.open(test_mask_fpath)

# Run prediction
masks = model.predict(test_image)
masks = torch.stack([masks == class_id for class_id in masks.unique()]).detach().cpu().numpy()
pred_mask = Image.fromarray(masks[-1])  # Tactile paving class (not background)

# Visualize
plt.figure(figsize=(12, 4), dpi=150)
plt.subplot(1, 3, 1)
plt.axis("off")
plt.imshow(test_image)
plt.title("Input Image")
plt.subplot(1, 3, 2)
plt.axis("off")
plt.imshow(gt_mask, cmap="gray")
plt.title("Ground Truth")
plt.subplot(1, 3, 3)
plt.axis("off")
plt.imshow(pred_mask, cmap="gray")
plt.title("Prediction")
plt.tight_layout()
plt.show()

## 6. Evaluation

Evaluate the model on the full test set.

In [ ]:
import sys
from tqdm import tqdm

sys.path.insert(0, "..")
from evaluation.metrics import compute_binary_metrics, aggregate_metrics

all_metrics = []

for img_name in tqdm(sorted(os.listdir(TEST_IMAGES)), desc="Evaluating"):
    if not img_name.endswith((".jpg", ".png")):
        continue

    img_path = os.path.join(TEST_IMAGES, img_name)
    mask_path = os.path.join(TEST_MASKS, img_name.replace(".jpg", ".png"))
    if not os.path.exists(mask_path):
        continue

    test_img = Image.open(img_path).convert("RGB")
    gt = np.array(Image.open(mask_path).convert("L"))

    pred = model.predict(test_img)
    pred = torch.stack([pred == c for c in pred.unique()]).detach().cpu().numpy()
    pred_binary = (pred[-1].astype(np.uint8) * 255)
    pred_resized = np.array(Image.fromarray(pred_binary).resize((gt.shape[1], gt.shape[0]), Image.NEAREST))

    metrics = compute_binary_metrics(gt, pred_resized)
    all_metrics.append(metrics)

avg = aggregate_metrics(all_metrics)
print(f"\nResults over {len(all_metrics)} test images:")
for k, v in avg.items():
    print(f"  {k}: {v:.4f}")